Connect to drive

In [2]:
cd

/root


In [4]:
ls

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Git clone the NCRFPP git repo

In [6]:
import os
if not os.path.exists('NCRFpp'):
    !git clone https://github.com/jiesutd/NCRFpp.git
else:
    print('Repository already exists.')

Cloning into 'NCRFpp'...
remote: Enumerating objects: 768, done.
remote: Total 768 (delta 0), reused 0 (delta 0), pack-reused 768 (from 1)
Receiving objects: 100% (768/768), 6.89 MiB | 16.88 MiB/s, done.
Resolving deltas: 100% (484/484), done.


In [7]:
ls

NCRFpp/


Add requirements.py and install the required libraries

In [8]:
with open('requirements.txt', 'w') as f:
    f.write('torch\nnumpy\n')
!pip install -r requirements.txt

Unzip the data file into project directory

In [9]:
!unzip -o "/content/drive/My Drive/MyAgriNER/data.zip" -d /NCRFpp

Archive:  /content/drive/My Drive/MyAgriNER/data.zip
   creating: /NCRFpp/data/
  inflating: /NCRFpp/data/bio_syl.conll  
  inflating: /NCRFpp/data/bioes.conll  
  inflating: /NCRFpp/data/bioes_syl.conll  
  inflating: /NCRFpp/data/bio.conll  


check if the data files are there

In [10]:
ls /NCRFpp/data

bio.conll  bioes.conll  bioes_syl.conll  bio_syl.conll


Slplit files into folder with train, dev and test

In [11]:
import os
import random
import glob

def split_conll_file(file_path, train_ratio=0.8, dev_ratio=0.1):
    if not os.path.exists(file_path):
        print(f"File {file_path} not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    # CoNLL files usually separate sentences/sequences with empty lines
    sequences = content.split('\n\n')
    random.seed(42)  # For reproducibility
    random.shuffle(sequences)

    total = len(sequences)
    train_end = int(total * train_ratio)
    dev_end = train_end + int(total * dev_ratio)

    train_data = sequences[:train_end]
    dev_data = sequences[train_end:dev_end]
    test_data = sequences[dev_end:]

    # Create a subfolder based on the filename
    file_name = os.path.basename(file_path)
    folder_name = file_name.replace('.conll', '')
    target_dir = os.path.join(os.path.dirname(file_path), folder_name)
    os.makedirs(target_dir, exist_ok=True)

    splits = {
        f'train.{file_name}': train_data,
        f'dev.{file_name}': dev_data,
        f'test.{file_name}': test_data
    }

    for out_name, data in splits.items():
        out_path = os.path.join(target_dir, out_name)
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(data) + '\n')
        print(f"Saved {len(data)} sequences to {out_path}")

# Dynamically process all .conll files in the directory that are not already splits
data_dir = '/NCRFpp/data'
conll_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
               if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for file in conll_files:
    split_conll_file(file)

Saved 5799 sequences to /NCRFpp/data/bio_syl/train.bio_syl.conll
Saved 724 sequences to /NCRFpp/data/bio_syl/dev.bio_syl.conll
Saved 726 sequences to /NCRFpp/data/bio_syl/test.bio_syl.conll
Saved 5640 sequences to /NCRFpp/data/bio/train.bio.conll
Saved 705 sequences to /NCRFpp/data/bio/dev.bio.conll
Saved 705 sequences to /NCRFpp/data/bio/test.bio.conll
Saved 5799 sequences to /NCRFpp/data/bioes/train.bioes.conll
Saved 724 sequences to /NCRFpp/data/bioes/dev.bioes.conll
Saved 726 sequences to /NCRFpp/data/bioes/test.bioes.conll
Saved 5799 sequences to /NCRFpp/data/bioes_syl/train.bioes_syl.conll
Saved 724 sequences to /NCRFpp/data/bioes_syl/dev.bioes_syl.conll
Saved 726 sequences to /NCRFpp/data/bioes_syl/test.bioes_syl.conll


In [15]:
import os
import glob

def create_config(name, train_path, dev_path, test_path):
    config_content = f"""
### use # to comment out the configure item

### I/O ###
train_dir=/NCRFpp/{train_path}
dev_dir=/NCRFpp/{dev_path}
test_dir=/NCRFpp/{test_path}
model_dir=/NCRFpp/models/{name}
#word_emb_dir=data/sample.word.emb

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=50
char_emb_dim=30

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=ADAM
iteration=30
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.5
lstm_layer=1
bilstm=True
learning_rate=0.015
lr_decay=0.05
momentum=0
l2=1e-8
gpu
#clip=
"""
    # Save config in the root NCRFpp directory
    config_path = f"/NCRFpp/demo.{name}.train.config"
    with open(config_path, 'w') as f:
        f.write(config_content.strip())
    print(f"Created config: {config_path}")

# Dynamic detection of files in data directory
data_dir = '/NCRFpp/data'
os.makedirs('/NCRFpp/models', exist_ok=True)

# Find all original conll names (e.g., bio.conll, bio_syl.conll)
conll_files = [f for f in os.listdir(data_dir) if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for base_file in conll_files:
    name = base_file.replace('.conll', '')

    # Construct paths relative to the execution context (NCRFpp folder)
    # Now pointing to the subfolders created by the splitter
    train_p = f"data/{name}/train.{base_file}"
    dev_p = f"data/{name}/dev.{base_file}"
    test_p = f"data/{name}/test.{base_file}"

    # Verify files exist before creating config
    if all(os.path.exists(os.path.join('/NCRFpp', p)) for p in [train_p, dev_p, test_p]):
        create_config(name, train_p, dev_p, test_p)
    else:
        print(f"Skipping {name}: Missing split files in subfolder {name}.")

Created config: /NCRFpp/demo.bio_syl.train.config
Created config: /NCRFpp/demo.bio.train.config
Created config: /NCRFpp/demo.bioes.train.config
Created config: /NCRFpp/demo.bioes_syl.train.config


In [13]:
cat /NCRFpp/demo.bioes.train.config

### use # to comment out the configure item

### I/O ###
train_dir=/NCRFpp/data/bioes/train.bioes.conll
dev_dir=/NCRFpp/data/bioes/dev.bioes.conll
test_dir=/NCRFpp/data/bioes/test.bioes.conll
model_dir=/NCRFpp/models/bioes
#word_emb_dir=data/sample.word.emb

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=50
char_emb_dim=30

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=ADAM
iteration=1
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.5
lstm_layer=1
bilstm=True
learning_rate=0.015
lr_decay=0.05
momentum=0
l2=1e-8
gpu
#clip=

In [14]:
!python NCRFpp/main.py --config /NCRFpp/demo.bio.train.config

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 10757
     Char  alphabet size: 96
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/bio/train.bio.conll
     Dev    file directory: /NCRFpp/data/bio/dev.bio.conll
     Test   file directory: /NCRFpp/data/bio/test.bio.conll
     Raw    file directory: None
     Dset   file directory: None
     Model  file directory: /NCRFpp/models/bio
     Loadmodel   directory: None
     Decode file directory: None
 

In [16]:
!python NCRFpp/main.py --config /NCRFpp/demo.bioes.train.config

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 11065
     Char  alphabet size: 101
     Label alphabet size: 139
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/bioes/train.bioes.conll
     Dev    file directory: /NCRFpp/data/bioes/dev.bioes.conll
     Test   file directory: /NCRFpp/data/bioes/test.bioes.conll
     Raw    file directory: None
     Dset   file directory: None
     Model  file directory: /NCRFpp/models/bioes
     Loadmodel   directory: None
     Decode file 

### Decoding Configuration Generator
This script generates `.decode.config` files for testing. It assumes that training has completed and produced model files in the `/NCRFpp/models/` directory.

In [17]:
import os
import glob

def create_decode_config(name, test_path, model_path, dset_path):
    # Ensure output directory exists
    os.makedirs('/NCRFpp/output', exist_ok=True)

    decode_content = f"""
### I/O ###
status=decode
raw_dir=/NCRFpp/{test_path}
decode_dir=/NCRFpp/output/{name}.test.out
dset_dir={dset_path}
load_model_dir={model_path}

nbest=10
#gpu
"""
    config_path = f"/NCRFpp/demo.{name}.decode.config"
    with open(config_path, 'w') as f:
        f.write(decode_content.strip())
    print(f"Created decode config: {config_path}")

# Updated detection logic for models in the root models folder
models_root = '/NCRFpp/models'
if os.path.exists(models_root):
    # Find all .dset files to identify trained datasets
    dset_paths = glob.glob(os.path.join(models_root, "*.dset"))

    for dset_path in dset_paths:
        # e.g., /NCRFpp/models/bio.dset -> name = 'bio'
        name = os.path.basename(dset_path).replace('.dset', '')

        # Find matching models for this name (e.g., bio.0.model, bio.1.model)
        model_files = sorted(glob.glob(os.path.join(models_root, f"{name}.*.model")))

        if model_files:
            # Use the latest model and the standard test path
            test_p = f"data/{name}/test.{name}.conll"
            create_decode_config(name, test_p, model_files[-1], dset_path)
        else:
            print(f"No models found for {name} in {models_root}")
else:
    print(f"Models directory {models_root} does not exist.")

Created decode config: /NCRFpp/demo.bioes.decode.config
Created decode config: /NCRFpp/demo.bio.decode.config


In [18]:
ls /NCRFpp

data/                     demo.bioes_syl.train.config  demo.bio.train.config
demo.bio.decode.config    demo.bioes.train.config      models/
demo.bioes.decode.config  demo.bio_syl.train.config    output/


In [19]:
ls /NCRFpp/models

bio.0.model    bioes.12.model  bioes.1.model   bioes.3.model
bio.dset       bioes.13.model  bioes.20.model  bioes.dset
bioes.0.model  bioes.14.model  bioes.2.model


In [20]:
!python NCRFpp/main.py --config /NCRFpp/demo.bioes.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/bioes/test.bioes.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 11065
     Char  alphabet size: 101
     Label alphabet size: 139
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/bioes/train.bioes.conll
     Dev    file directory: /NCRFpp/data/bioes/dev.bioes.conll
     Test   file directory: /NCRFpp/data/bioes/test.bioes.conll
     Raw    file directory: /NCRFpp/data/bioes/test.bioes.conll
     Dset   file directory: /NCRFpp/models/bioes.dset
     Model  file directory: 

In [177]:
ls /NCRFpp/output/

bioes.test.out  bio.test.out


In [21]:
!head /NCRFpp/output/bioes.test.out -n 200

# 0.1794 0.1648 0.1406 0.1236 0.0794 0.0719 0.0661 0.0604 0.0574 0.0564
ဩဘာ O O O O O S-ABIOD S-ABIOD S-SYM O S-ABIOD
က O O O O O O O O O O
တော့ O O O O O O O O O O
၊ O O O O O O O O O O
ဘက် O O O O O O O O O O
တာ O O O O O O O O O O
နှင့် O O O O O O O O O O
အလန်း O S-FERT S-PESTI S-FUNG S-NUT O S-FERT O S-DIS S-PESTI
ဖြန်း O O O O O O O O O O
လို O O O O O O O O O O
ပြော O O O O O O O O O O
တယ် O O O O O O O O O O
။ O O O O O O O O O O

# 0.8812 0.0295 0.0229 0.0147 0.0105 0.0100 0.0095 0.0080 0.0076 0.0060
ဆောင်း B-SEASON B-SEASON B-SEASON B-SEASON B-SEASON B-SEASON B-SEASON S-SEASON B-SEASON B-SEASON
ရာသီ E-SEASON E-SEASON E-SEASON E-SEASON E-SEASON E-SEASON E-SEASON E-SEASON E-SEASON E-SEASON
မှာ O O O O O O O O O O
ဘို B-CROP B-CROP B-CROP B-CROP B-CROP B-CROP B-CROP B-CROP B-CROP B-CROP
ကိတ် I-CROP I-CROP I-CROP I-CROP I-CROP E-CROP I-CROP I-CROP I-CROP I-CROP
ပဲ E-CROP E-CROP E-CROP E-CROP E-CROP O E-CROP E-CROP E-CROP E-CROP
ကို O O O O O O O O O O
ကြဲပက် S-FARM_OP O S-FARM_OP

In [22]:
import shutil
from datetime import datetime

# Define the destination path in Google Drive
drive_export_path = '/content/drive/My Drive/MyAgriNER/export'
os.makedirs(drive_export_path, exist_ok=True)

# Source directories to backup
sources = {
    'models': '/NCRFpp/models',
    'output': '/NCRFpp/output',
    'configs': '/NCRFpp/*.config'
}

print(f"Starting backup to {drive_export_path}...")

# Copy models and output folders
for folder in ['models', 'output']:
    src = f'/NCRFpp/{folder}'
    dst = os.path.join(drive_export_path, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✓ Copied {folder} to Drive.")

# Copy config files specifically
import glob
for config_file in glob.glob('/NCRFpp/*.config'):
    shutil.copy(config_file, drive_export_path)

print("\nBackup complete! Your models, outputs, and configs are now safe in your Google Drive.")

Starting backup to /content/drive/My Drive/MyAgriNER/export...
✓ Copied models to Drive.
✓ Copied output to Drive.

Backup complete! Your models, outputs, and configs are now safe in your Google Drive.
